# Exercise 1: Quantum Route Optimization

## Learning Objectives
In this exercise, you will:
1. Understand the Traveling Salesman Problem (TSP) and its complexity
2. Learn how to formulate TSP as a QUBO problem
3. Convert QUBO to an Ising Hamiltonian
4. Implement the Variational Quantum Eigensolver (VQE) algorithm
5. Compare quantum and classical solutions

## Introduction

The **Traveling Salesman Problem (TSP)** is a classic NP-Complete problem in computer science:

> *Given a list of cities and the distances between each pair of cities, what is the shortest possible route that visits each city exactly once and returns to the origin city?*

For large numbers of cities, this problem becomes computationally intractable for classical computers. Quantum computing offers a potential advantage through algorithms like VQE.

### Your Task
Complete the missing code sections marked with `# TODO` throughout this notebook.

## Part 1: Environment Setup

First, let's install and import the necessary dependencies.

In [ ]:
# Install packages in the current Jupyter kernel

import sys
!{sys.executable} -m pip install qiskit==2.4.1
!{sys.executable} -m pip install qiskit-aer==0.17.2
!{sys.executable} -m pip install qiskit-algorithms==0.4.0
!{sys.executable} -m pip install qiskit-ibm-runtime==0.47.0
!{sys.executable} -m pip install qiskit-machine-learning==0.9.0
!{sys.executable} -m pip install qiskit-optimization==0.7.0
!{sys.executable} -m pip install networkx
!{sys.executable} -m pip install scipy

In [ ]:
# Check Qiskit version
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

## Part 2: Helper Functions

We'll need functions to visualize our graphs and solutions.

In [ ]:
# Utility function to draw an input graph
def draw_graph(G, colors, pos):
    default_axes = plt.axes(frameon=True)
    nx.draw_networkx(G, node_color=colors, node_size=600, alpha=0.8, ax=default_axes, pos=pos)
    edge_labels = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(G, pos=pos, edge_labels=edge_labels)
    plt.title("Cities and route lengths", fontsize=12)


# Utility function to highlight a graph path on a given input graph
def draw_tsp_solution(G, order, colors, pos, title):
    G2 = nx.DiGraph()
    G2.add_nodes_from(G)
    n = len(order)
    for i in range(n):
        j = (i + 1) % n
        G2.add_edge(order[i], order[j], weight=G[order[i]][order[j]]["weight"])
    default_axes = plt.axes(frameon=True)
    nx.draw_networkx(G2, node_color=colors, edge_color="b", node_size=600, alpha=0.8, ax=default_axes, pos=pos)
    edge_labels = nx.get_edge_attributes(G2, "weight")
    nx.draw_networkx_edge_labels(G2, pos, font_color="b", edge_labels=edge_labels)
    plt.title("Best route found - "+title, fontsize=12)

## Part 3: Problem Definition

### TODO 2: Create a TSP Problem Instance

Create a random TSP instance with 4 cities and visualize it.

**Hint**: Use `Tsp.create_random_instance(n)` from qiskit_optimization.applications

In [ ]:
# TODO: Define the number of cities
n = # YOUR CODE HERE

# TODO: Create a random TSP instance
tsp_problem = # YOUR CODE HERE
G = tsp_problem.graph

# Get adjacency matrix
adj_matrix = nx.to_numpy_array(G)
print("Distance matrix:\n", adj_matrix)

# Define colors for visualization
base_colors = ["red", "blue", "green", "orange", "purple", "cyan", "magenta", "yellow"]
colors = base_colors[:n]
pos = nx.spring_layout(G)

# Draw the graph
fig, ax = plt.subplots(1, figsize=(7, 5))
draw_graph(G, colors, pos)

## Part 4: QUBO Formulation

### TODO 3: Convert TSP to QUBO

The TSP can be formulated as a Quadratic Unconstrained Binary Optimization (QUBO) problem.

**Question**: Why do we need to convert TSP to QUBO format?

**Your answer here**: 

---

Complete the code to convert the TSP to a quadratic program.

In [ ]:
from qiskit_optimization.converters import QuadraticProgramToQubo

# TODO: Convert TSP to quadratic program
qp = # YOUR CODE HERE (use tsp_problem method)

print("QUBO formulation:")
print(qp)

## Part 5: Ising Hamiltonian

### TODO 4: Convert QUBO to Ising Hamiltonian

To use quantum algorithms, we need to convert the QUBO to an Ising Hamiltonian.

**Hint**: Use `QuadraticProgramToQubo` converter and then convert to Ising operator.

In [ ]:
# TODO: Import necessary converters
# from qiskit_optimization.converters import ...

# TODO: Convert to QUBO format
qp2qubo = # YOUR CODE HERE
qubo = # YOUR CODE HERE

# TODO: Convert to Ising Hamiltonian
ising_hamiltonian, offset = # YOUR CODE HERE

print(f"Offset: {offset}")
print(f"Ising Hamiltonian: {ising_hamiltonian}")

## Part 6: Quantum Circuit - Ansatz

### Create the Ansatz

The ansatz is the parameterized quantum circuit that VQE will optimize.

**Question**: What is an ansatz and why is it important in VQE?

**Your answer here**:

---

**Hint**: Use `EfficientSU2` with appropriate number of qubits and repetitions.

In [ ]:
from qiskit.circuit import QuantumCircuit, ParameterVector

# Initialization of the base Ansatz circuit
def base_circuit(QC, n, theta):
    theta1 = ParameterVector('theta2', 1)
    QC.x(0)
    QC.ry(theta[1], 1)
    QC.cz(0, 1)
    QC.ry(-theta[1], 1)
    QC.cx(1, 0)
    QC.cx(1, n)
    QC.cx(0, n + 1) 
    
# y axis rotation and controlled gates to encode all possible solutions of the problem in the Ansatz circuit
def W_circuit(QC, n, q1n, theta):
    QC.x(q1n)
    for j in range(q1n + 1, q1n + n, 1):
        QC.ry(theta[j - 1], j)
        QC.cz(j - 1, j)
        QC.ry(-theta[j - 1], j)
    for j in range(q1n + 1, q1n + n, 1):
        QC.cx(j, j - 1)

num_qubits = n ** 2
qc = QuantumCircuit(num_qubits)
phi = ParameterVector('phi', num_qubits)

# The following lines define a custom Ansatz for the TSP using the above base_circuit and W_circuit 
base_circuit(qc, n, phi)

for i in range(3, n + 1, 1):
    W_circuit(qc, i, n * (i - 1), phi)
for k in range(3, n + 1, 1):
    for v in range(1, k, 1):
        for p in range(1, k, 1):
            qc.cswap(n * (k - 1) + v - 1, n * (p - 1) + n - (n - k) - 1, n * (p - 1) + v - 1)
qc.measure_all()

ansatz = qc


## Part 7: Backend Configuration

### TODO 6: Configure the Quantum Backend

For this exercise, we'll use a simulator. In production, you could use real quantum hardware.

**Note**: If you have IBM Quantum credentials, you can uncomment and configure the runtime service.

In [ ]:
# TODO: Import and configure backend
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

# Use simulator for this exercise
backend = AerSimulator()
print("Using AerSimulator for local simulation")

# Optional: Configure IBM Quantum Runtime (requires credentials)
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService(
#     channel="ibm_quantum",
#     token="YOUR_TOKEN_HERE"
# )
# backend = service.least_busy(operational=True, simulator=False)

## Part 8: Circuit Transpilation

### TODO 7: Transpile the Circuit

Transpilation adapts the circuit to the backend's constraints.

**Hint**: Use `transpile()` with optimization_level=3

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import transpile

# TODO: Transpile the ansatz
isa_ansatz = # YOUR CODE HERE

print(f"Original circuit depth: {ansatz.depth()}")
print(f"Transpiled circuit depth: {isa_ansatz.depth()}")

## Part 9: Cost Function

### TODO 8: Implement the Cost Function

The cost function evaluates how good a particular set of parameters is.

**Hint**: The cost function should:
1. Bind parameters to the ansatz
2. Run the estimator
3. Return the expectation value

In [ ]:
# Global iteration counter
iteration = 0

def cost_func(params, ansatz, hamiltonian, estimator):
    """
    Cost function for VQE optimization.
    
    Args:
        params: Circuit parameters
        ansatz: Parameterized quantum circuit
        hamiltonian: Ising Hamiltonian
        estimator: Qiskit Estimator
    
    Returns:
        Expectation value (cost)
    """
    global iteration
    
    # TODO: Implement cost function
    # 1. Run estimator with (ansatz, hamiltonian, params)
    # 2. Extract expectation value from result
    # 3. Print progress
    # 4. Return cost
    
    # YOUR CODE HERE
    


## Part 10: Run VQE Optimization

### TODO 9: Execute VQE

Now we'll run the VQE algorithm to find the optimal parameters.

**Hint**: Use `scipy.optimize.minimize` with COBYLA method

In [ ]:
from qiskit_algorithms.utils import validate_initial_point
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

# TODO: Prepare for optimization
num_params = # YOUR CODE HERE
x0 = validate_initial_point(None, isa_ansatz)
hamiltonian_isa = ising_hamiltonian.apply_layout(isa_ansatz.layout)

# TODO: Run optimization
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 3000
    
    # YOUR CODE HERE: Use minimize() to optimize
    res = # YOUR CODE HERE
    
    session.close()

print("\nOptimization complete!")
print(f"Final cost: {res.fun}")
print(f"Success: {res.success}")

## Part 11: Extract and Interpret Results

### TODO 10: Get the Solution

Extract the solution from the optimized parameters and interpret it.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

# TODO: Sample the optimized circuit
with Session(backend=backend) as session:
    sampler = Sampler(mode=session)
    sampler.options.default_shots = 10000
    
    # YOUR CODE HERE: Bind parameters and run sampler
    
    session.close()

# TODO: Interpret results
# Extract the most probable bitstring
# Convert to TSP solution
# Calculate total distance

print("\nQuantum Solution:")
# YOUR CODE HERE

## Part 12: Visualize the Solution

### TODO 11: Draw the Optimal Route

In [ ]:
# TODO: Visualize the quantum solution
fig, ax = plt.subplots(1, figsize=(7, 5))
# YOUR CODE HERE: Use draw_tsp_solution()

## Part 13: Classical Comparison

### TODO 12: Compare with Classical Solution

Solve the same problem using a classical algorithm and compare results.

In [ ]:
# TODO: Solve using classical method
# Hint: Use networkx or implement brute force for small n

# YOUR CODE HERE

print("\nClassical Solution:")
# YOUR CODE HERE

## Part 14: Analysis and Reflection

### Questions for Discussion

1. **How does the quantum solution compare to the classical solution?**
   
   Your answer:

2. **What are the advantages and limitations of using VQE for TSP?**
   
   Your answer:

3. **How would the problem scale with more cities?**
   
   Your answer:

4. **What role does the ansatz play in the solution quality?**
   
   Your answer:

5. **How might noise in real quantum hardware affect the results?**
   
   Your answer:

## Bonus Challenges

If you complete the main exercise, try these extensions:

1. **Increase the problem size**: Try with 5 or 6 cities and observe the computational requirements
2. **Different ansatz**: Experiment with different ansatz circuits (e.g., RealAmplitudes, TwoLocal)
3. **Optimization methods**: Try different classical optimizers (SLSQP, SPSA, etc.)
4. **Real hardware**: If you have access, run on actual quantum hardware and compare with simulation
5. **Noise simulation**: Add noise models to the simulator to see how it affects results

## Conclusion

Congratulations! You've completed the Quantum Route Optimization exercise. You've learned:

- How to formulate TSP as a quantum optimization problem
- The basics of VQE algorithm
- How to use Qiskit for quantum computing
- The relationship between classical and quantum approaches

### Next Steps

- Explore other quantum algorithms (QAOA, Grover's algorithm)
- Study more complex optimization problems
- Learn about quantum error correction
- Investigate hybrid quantum-classical algorithms